In [1]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

/home/teaching/miniconda3/envs/dl45/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_model(base_model_name, adapter_path=None, quantize=True):
    
    if quantize:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
    else:
        bnb_config = None

    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    # If LoRA adapter exists → load it
    if adapter_path:
        model = PeftModel.from_pretrained(base_model, adapter_path)
        model = model.merge_and_unload()
    else:
        model = base_model

    return model, tokenizer

In [3]:
def gen_prompt(content):
    return f"""
You are an AI that converts raw job descriptions into structured JSON.

Return ONLY JSON.

INPUT:
{content}

OUTPUT:
"""

def edit_prompt(current_json, instruction):
    return f"""
You are an AI that updates job description JSON.

Modify ONLY necessary fields.

Existing JSON:
{json.dumps(current_json, indent=2)}

Instruction:
{instruction}

Return updated JSON only.
"""

In [4]:
import re

def extract_first_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            return None
    return None

In [5]:
def generate_jd(model, tokenizer, user_input):
    
    prompt = gen_prompt(user_input)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        repetition_penalty=1.2
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return extract_first_json(text)

In [6]:
def edit_jd(model, tokenizer, state, instruction):

    prompt = edit_prompt(state, instruction)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        repetition_penalty=1.2
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return extract_first_json(text)

In [7]:
def route(state):
    if state is None:
        return "generate"
    else:
        return "edit"

In [8]:
class JDModel:
    
    def __init__(self, gen_model, gen_tokenizer, edit_model, edit_tokenizer):
        self.gen_model = gen_model
        self.gen_tokenizer = gen_tokenizer
        
        self.edit_model = edit_model
        self.edit_tokenizer = edit_tokenizer
        
        self.state = None  # stores current JSON

    def __call__(self, user_input):
        
        mode = route(self.state)

        if mode == "generate":
            print("\n[MODE] GENERATION")
            self.state = generate_jd(self.gen_model, self.gen_tokenizer, user_input)

        else:
            print("\n[MODE] EDITING")
            updated = edit_jd(self.edit_model, self.edit_tokenizer, self.state, user_input)
            
            if updated:
                self.state = updated

        return self.state

In [9]:
# 🔹 Generation model
gen_model, gen_tokenizer = load_model(
    base_model_name="./models/gemma-2b-it",
    adapter_path="./models/gemma-2b-it-fine-tuned",   # your generation LoRA
)

# 🔹 Editing model
edit_model, edit_tokenizer = load_model(
    base_model_name="models/gemma-2b-it",
    adapter_path="./models/gemma-2b-it-fine-tuned-edit",  # your editing LoRA
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:50<00:00, 25.09s/it]
/home/teaching/miniconda3/envs/dl45/lib/python3.11/site-packages/peft/tuners/lora/bnb.py:355: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:47<00:00, 23.67s/it]


In [10]:
jd_model = JDModel(gen_model, gen_tokenizer, edit_model, edit_tokenizer)

while True:
    user_input = input("\n>> ")

    if user_input.lower() == "exit":
        break

    output = jd_model(user_input)

    print("\nUPDATED JD:\n", json.dumps(output, indent=2))

The 'batch_size' attribute of HybridCache is deprecated and will be removed in v4.49. Use the more precisely named 'self.max_batch_size' attribute instead.



[MODE] GENERATION

UPDATED JD:
 {
  "job_title": "Python Developer",
  "experience": "2+",
  "skills": [
    "python"
  ],
  "work_type": "backend"
}


In [11]:
edit_model.save_pretrained("./final_edit_model")
edit_tokenizer.save_pretrained("./final_edit_model")

('./final_edit_model/tokenizer_config.json',
 './final_edit_model/special_tokens_map.json',
 './final_edit_model/tokenizer.json')